In [1]:
import sys
sys.path.append('..')

import json
import joblib
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from src.explain import build_baseline

X_counts = np.load('../data/features/X_counts_full.npy')
y = np.load('../data/features/y_full.npy')

X_normal = X_counts[y == 0]
print("Training on:", X_normal.shape)

Training on: (558223, 45)


In [2]:
pca = PCA(n_components=5, random_state=42)
pca.fit(X_normal)

reconstructed = pca.inverse_transform(pca.transform(X_normal))
errors = np.mean((X_normal - reconstructed) ** 2, axis=1)
pca_threshold = float(np.percentile(errors, 97))

joblib.dump(pca, '../models/pca.joblib')
print("PCA threshold:", round(pca_threshold, 6))

PCA threshold: 7.3e-05


In [3]:
iso = IsolationForest(contamination=0.03, random_state=42)
iso.fit(X_normal)
joblib.dump(iso, '../models/isolation_forest.joblib')

rng = np.random.default_rng(42)
idx = rng.choice(len(X_normal), size=50_000, replace=False)
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.03, novelty=True)
lof.fit(X_normal[idx])
joblib.dump(lof, '../models/lof.joblib')

print("Saved all three models")

Saved all three models


In [4]:
templates = pd.read_csv('../data/parsed/templates_full.csv')
event_names = templates['EventId'].tolist()

baseline = build_baseline(X_counts, y)
np.save('../models/baseline.npy', baseline)

metadata = {
    'event_names': [int(e) for e in event_names],
    'templates': {int(r.EventId): r.EventTemplate for r in templates.itertuples()},
    'pca_threshold': pca_threshold,
    'anomaly_rate': float(y.mean()),
    'trained_sessions': int(len(X_normal)),
}

with open('../models/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved metadata with", len(event_names), "events")

Saved metadata with 45 events
